In [ ]:
import os
# 读取环境变量
def load_env(file_path=".env"):
    env_vars = {}
    with open(file_path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            key, value = line.split("=", 1)
            env_vars[key] = value
    return env_vars

env = load_env(r"D:\Code\Qiniu\Python\api.env")
print(env)


# -*- coding: utf-8 -*-
import json
import os
import base64
from typing import Optional, Dict, Any
from tencentcloud.common import credential
from tencentcloud.common.profile.client_profile import ClientProfile
from tencentcloud.common.profile.http_profile import HttpProfile
from tencentcloud.common.exception.tencent_cloud_sdk_exception import TencentCloudSDKException
from tencentcloud.ai3d.v20250513 import ai3d_client, models


class TencentAI3DGenerator:
    """
    腾讯云AI 3D生成服务类
    支持文本生成3D和图片生成3D功能
    """
    
    def __init__(self, secret_id: str = None, secret_key: str = None, region: str = "ap-guangzhou"):
        """
        初始化AI 3D生成器
        
        Args:
            secret_id: 腾讯云SecretId，如果不提供会从环境变量TENCENTCLOUD_SECRET_ID获取
            secret_key: 腾讯云SecretKey，如果不提供会从环境变量TENCENTCLOUD_SECRET_KEY获取
            region: 服务区域，默认为ap-guangzhou
        """
        self.secret_id = secret_id or os.getenv("TENCENTCLOUD_SECRET_ID")
        self.secret_key = secret_key or os.getenv("TENCENTCLOUD_SECRET_KEY")
        self.region = region
        
        if not self.secret_id or not self.secret_key:
            raise ValueError("请提供有效的secret_id和secret_key，或设置环境变量TENCENTCLOUD_SECRET_ID和TENCENTCLOUD_SECRET_KEY")
        
        self._init_client()
    
    def _init_client(self):
        """初始化腾讯云客户端"""
        try:
            # 实例化一个认证对象
            cred = credential.Credential(self.secret_id, self.secret_key)
            
            # 实例化一个http选项
            httpProfile = HttpProfile()
            httpProfile.endpoint = "ai3d.tencentcloudapi.com"
            
            # 实例化一个client选项
            clientProfile = ClientProfile()
            clientProfile.httpProfile = httpProfile
            
            # 实例化AI3D客户端
            self.client = ai3d_client.Ai3dClient(cred, self.region, clientProfile)
            
        except Exception as e:
            raise RuntimeError(f"初始化腾讯云客户端失败: {str(e)}")
    
    def text_to_3d(self, 
                   prompt: str, 
                   result_format: str = "OBJ", 
                   enable_pbr: bool = False) -> Dict[str, Any]:
        """
        文本生成3D模型
        
        Args:
            prompt: 文本描述，用于生成3D模型
            result_format: 输出格式，可选值：OBJ, GLB, MP4, STL, FBX, USDZ
            enable_pbr: 是否启用PBR材质
            
        Returns:
            Dict: API响应结果
        """
        try:
            req = models.SubmitHunyuanTo3DRapidJobRequest()
            
            params = {
                "Action": "SubmitHunyuanTo3DRapidJob",
                "Region": self.region,
                "Version": "2025-05-13",
                "Prompt": prompt,
                "ResultFormat": result_format.upper(),
                "EnablePBR": enable_pbr
            }
            
            req.from_json_string(json.dumps(params))
            
            # 发送请求
            resp = self.client.SubmitHunyuanTo3DRapidJob(req)
            
            return json.loads(resp.to_json_string())
            
        except TencentCloudSDKException as e:
            raise RuntimeError(f"文本生成3D请求失败: {e.message}")
        except Exception as e:
            raise RuntimeError(f"文本生成3D处理失败: {str(e)}")
    
    def image_to_3d_by_url(self, 
                           image_url: str, 
                           prompt: str = "", 
                           result_format: str = "OBJ", 
                           enable_pbr: bool = False) -> Dict[str, Any]:
        """
        通过图片URL生成3D模型
        
        Args:
            image_url: 输入图片URL（需要是公网可访问的URL）
            prompt: 可选的文本描述
            result_format: 输出格式，可选值：OBJ, GLB, MP4, STL, FBX, USDZ
            enable_pbr: 是否启用PBR材质
            
        Returns:
            Dict: API响应结果
        """
        try:
            req = models.SubmitHunyuanTo3DRapidJobRequest()
            
            params = {
                "Action": "SubmitHunyuanTo3DRapidJob",
                "Region": self.region,
                "Version": "2025-05-13",
                "ImageUrl": image_url,
                "ResultFormat": result_format.upper(),
                "EnablePBR": enable_pbr
            }
            
            if prompt:
                params["Prompt"] = prompt
            
            req.from_json_string(json.dumps(params))
            
            # 发送请求
            resp = self.client.SubmitHunyuanTo3DRapidJob(req)
            
            return json.loads(resp.to_json_string())
            
        except TencentCloudSDKException as e:
            raise RuntimeError(f"图片生成3D请求失败: {e.message}")
        except Exception as e:
            raise RuntimeError(f"图片生成3D处理失败: {str(e)}")
    
    def image_to_3d_by_base64(self, 
                              image_base64: str, 
                              prompt: str = "", 
                              result_format: str = "OBJ", 
                              enable_pbr: bool = False) -> Dict[str, Any]:
        """
        通过Base64编码的图片生成3D模型
        
        Args:
            image_base64: Base64编码的图片数据
            prompt: 可选的文本描述
            result_format: 输出格式，可选值：OBJ, GLB, MP4, STL, FBX, USDZ
            enable_pbr: 是否启用PBR材质
            
        Returns:
            Dict: API响应结果
        """
        try:
            req = models.SubmitHunyuanTo3DRapidJobRequest()
            
            params = {
                "Action": "SubmitHunyuanTo3DRapidJob",
                "Region": self.region,
                "Version": "2025-05-13",
                "ImageBase64": image_base64,
                "ResultFormat": result_format.upper(),
                "EnablePBR": enable_pbr
            }
            
            if prompt:
                params["Prompt"] = prompt
            
            req.from_json_string(json.dumps(params))
            
            # 发送请求
            resp = self.client.SubmitHunyuanTo3DRapidJob(req)
            
            return json.loads(resp.to_json_string())
            
        except TencentCloudSDKException as e:
            raise RuntimeError(f"图片生成3D请求失败: {e.message}")
        except Exception as e:
            raise RuntimeError(f"图片生成3D处理失败: {str(e)}")
    
    def image_to_3d_by_file(self, 
                            image_path: str, 
                            prompt: str = "", 
                            result_format: str = "OBJ", 
                            enable_pbr: bool = False) -> Dict[str, Any]:
        """
        通过本地图片文件生成3D模型
        
        Args:
            image_path: 本地图片文件路径
            prompt: 可选的文本描述
            result_format: 输出格式，可选值：OBJ, GLB, MP4, STL, FBX, USDZ
            enable_pbr: 是否启用PBR材质
            
        Returns:
            Dict: API响应结果
        """
        if not os.path.exists(image_path):
            raise FileNotFoundError(f"图片文件不存在: {image_path}")
        
        try:
            # 读取图片并转换为Base64
            with open(image_path, 'rb') as f:
                image_data = f.read()
                image_base64 = base64.b64encode(image_data).decode('utf-8')
            
            return self.image_to_3d_by_base64(image_base64, prompt, result_format, enable_pbr)
            
        except Exception as e:
            raise RuntimeError(f"读取图片文件失败: {str(e)}")


class TaskStatus(Enum):
    """任务状态枚举"""
    WAIT = "WAIT"              # 等待中
    RUN = "RUN"                # 执行中
    FAIL = "FAIL"              # 任务失败
    DONE = "DONE"              # 任务成功
    UNKNOWN = "UNKNOWN"        # 未知状态


class TencentAI3DQueryClient:
    """
    腾讯云AI 3D任务查询客户端
    用于查询3D生成任务的状态和结果
    """
    
    def __init__(self, secret_id: str = None, secret_key: str = None, region: str = "ap-guangzhou"):
        """
        初始化查询客户端
        
        Args:
            secret_id: 腾讯云SecretId，如果不提供会从环境变量获取
            secret_key: 腾讯云SecretKey，如果不提供会从环境变量获取
            region: 服务区域，默认为ap-guangzhou
        """
        self.secret_id = secret_id or os.getenv("TENCENTCLOUD_SECRET_ID")
        self.secret_key = secret_key or os.getenv("TENCENTCLOUD_SECRET_KEY")
        self.region = region
        
        if not self.secret_id or not self.secret_key:
            raise ValueError("请提供有效的secret_id和secret_key，或设置环境变量")
        
        self._init_client()
    
    def _init_client(self):
        """初始化腾讯云客户端"""
        try:
            cred = credential.Credential(self.secret_id, self.secret_key)
            
            httpProfile = HttpProfile()
            httpProfile.endpoint = "ai3d.tencentcloudapi.com"
            
            clientProfile = ClientProfile()
            clientProfile.httpProfile = httpProfile
            
            self.client = ai3d_client.Ai3dClient(cred, self.region, clientProfile)
            
        except Exception as e:
            raise RuntimeError(f"初始化腾讯云客户端失败: {str(e)}")
    
    def query_task(self, job_id: str) -> Optional[Dict[str, Any]]:
        """
        查询单个任务状态和结果
        
        Args:
            job_id: 任务ID
            
        Returns:
            Dict: 任务状态和结果信息，失败时返回None
        """
        try:
            req = models.QueryHunyuanTo3DRapidJobRequest()
            req.JobId = job_id
            
            resp = self.client.QueryHunyuanTo3DRapidJob(req)
            result = json.loads(resp.to_json_string())
            
            return result
            
        except TencentCloudSDKException as e:
            print(f"查询任务失败 (JobId: {job_id}): {e.message}")
            return None
        except Exception as e:
            print(f"查询任务处理失败 (JobId: {job_id}): {str(e)}")
            return None
  



In [ ]:

# 初始化3D生成器
generator = TencentAI3DGenerator(env["TENCENTCLOUD_SECRET_ID"], env["TENCENTCLOUD_SECRET_KEY"])

try:
    # 文本生成3D示例
    print("=== 文本生成3D ===")
    result = generator.text_to_3d(
        prompt="一匹奔跑的骏马",
        result_format="STL",
        enable_pbr=False
    )
    print(json.dumps(result, indent=2, ensure_ascii=False))
    
    # 图片URL生成3D示例
    print("\n=== 图片URL生成3D ===")
    # result = generator.image_to_3d_by_url(
    #     image_url="https://example.com/horse.jpg",
    #     prompt="骏马",
    #     result_format="GLB"
    # )
    # print(json.dumps(result, indent=2, ensure_ascii=False))
    
    # 本地图片生成3D示例
    print("\n=== 本地图片生成3D ===")
    # result = generator.image_to_3d_by_file(
    #     image_path="./horse.jpg",
    #     prompt="骏马",
    #     result_format="MP4"
    # )
    # print(json.dumps(result, indent=2, ensure_ascii=False))
    
except Exception as e:
    print(f"错误: {e}")